In [106]:
import math

import numpy as np
import pandas as pd
import plotly.express as px
import random

In [107]:
pval = 1
pi = np.array([[pval], [1-pval]])

In [108]:
#inital request's probability of being valid given data, descriptors or input
pval = 1

#component performance
p_c = .75

pi = np.array([[pval], [1-pval]])
C = np.array([[p_c, 0], [1-p_c, 1]])

In [109]:
#observed component performance
C @ pi

array([[0.75],
       [0.25]])

In [110]:
#building 

rng = np.random.default_rng(seed=42)

n_components = 6


def build_component_matrix(p_c):
    return np.array(np.array([[p_c, 0], [1-p_c, 1]]))


component_ensemble = {}

for i in range(n_components):
    component_id = f"C{i + 1}"
    p_c = rng.uniform(low=0.5, high=0.8)
    component_ensemble[component_id] = build_component_matrix(p_c)


In [111]:
rng = np.random.default_rng(seed=7)

n_users = 100


def build_human_matrix(p_ac, p_ai):
    return np.array([[p_ac, p_ai], [1 - p_ac, 1 - p_ai]])


user_ensemble = {}

for i in range(n_users):
    user_id = f"u{i + 1}"
    p_ac = 1
    p_ai = 0
    user_ensemble[user_id] = build_human_matrix(p_ac, p_ai)

In [112]:
component_ensemble.keys()

dict_keys(['C1', 'C2', 'C3', 'C4', 'C5', 'C6'])

In [113]:
component_ensemble.keys()

dict_keys(['C1', 'C2', 'C3', 'C4', 'C5', 'C6'])

In [114]:
import numpy as np

def link_system_components(ensemble, component_list):
    result = np.eye(2)
    for key in component_list:
        result = np.matmul(result, ensemble[key])
        
    return result

In [115]:
#improvement rate measures as (1-c11)*r per execusion (cadence controlled in the simulation step)
def component_improvement_sprint(component_ensemble, component_list, improvement_rate):
    for key in component_list:
        improvement = (1-component_ensemble[key][0][0])*improvement_rate
        component_ensemble[key][0][0] += improvement
        component_ensemble[key][1][0] -= improvement
    return component_ensemble

In [116]:
def simulate_daily_activity(day, n_items, user_ensemble, component_ensemble):
    pi = np.array([[1],[0]])
    telemetry = []
    user_ids = list(user_ensemble.keys())
    component_ids = list(component_ensemble.keys())

    for _ in range(n_items):
        user = rng.choice(user_ids)
        system_inference_path = random.sample(component_ids, random.randint(2,6))
        composite_system_performance = link_system_components(component_ensemble, system_inference_path)
        accept_prob = float((user_ensemble[user] @ composite_system_performance @ pi)[0, 0])
        is_accepted = rng.random() < accept_prob
        telemetry.append(
            [day, user, is_accepted, system_inference_path]
        )

    return pd.DataFrame(
        telemetry,
        columns=[
            "Date",
            "User",
            "isAccepted",
            "systemInferencePath",
        ],
    )

In [117]:
simulate_daily_activity(1,1000,user_ensemble, component_ensemble)

,Date,User,isAccepted,systemInferencePath
0,1,u95,False,"[C6, C5]"
1,1,u63,False,"[C5, C6, C1]"
2,1,u84,False,"[C1, C6, C4, C2, C3]"
3,1,u23,False,"[C3, C5, C4, C6, C2, C1]"
4,1,u92,False,"[C1, C2]"
...,...,...,...,...
995,1,u40,False,"[C1, C2, C5, C4, C3]"
996,1,u16,False,"[C5, C1]"
997,1,u43,True,"[C2, C4, C6]"
998,1,u100,False,"[C5, C2, C6]"


In [118]:
n_items_per_day = 1000
simulation_duration = 365 + 90

historical_telemetry = []

for day_idx in range(simulation_duration):
    # system_correct_prob = min(system_quality_over_time(day_idx), 1.0)
    historical_telemetry.append(
        simulate_daily_activity(
            day=day_idx + 1,
            n_items=n_items_per_day,
            user_ensemble=user_ensemble,
            component_ensemble=component_ensemble,
        )
    )

    if (day_idx + 1) % 7 == 0:
        component_ensemble = component_improvement_sprint(component_ensemble, random.sample(list(component_ensemble.keys()), 1), .4)

historical_telemetry_df = pd.concat(historical_telemetry, ignore_index=True)
historical_telemetry_df.head()

,Date,User,isAccepted,systemInferencePath
0,1,u54,False,"[C6, C3, C5, C1, C2, C4]"
1,1,u65,False,"[C3, C6, C2, C1, C4, C5]"
2,1,u12,False,"[C3, C6, C2, C1, C4, C5]"
3,1,u19,True,"[C1, C2, C6]"
4,1,u89,False,"[C6, C5, C1, C3]"


In [119]:
historical_telemetry_df

,Date,User,isAccepted,systemInferencePath
0,1,u54,False,"[C6, C3, C5, C1, C2, C4]"
1,1,u65,False,"[C3, C6, C2, C1, C4, C5]"
2,1,u12,False,"[C3, C6, C2, C1, C4, C5]"
3,1,u19,True,"[C1, C2, C6]"
4,1,u89,False,"[C6, C5, C1, C3]"
...,...,...,...,...
454995,455,u72,True,"[C1, C5, C4, C2, C6, C3]"
454996,455,u44,True,"[C2, C4]"
454997,455,u55,True,"[C6, C4, C1, C5, C3]"
454998,455,u9,False,"[C4, C1, C3, C5, C6, C2]"


In [120]:
observed_daily_accepts = (
    historical_telemetry_df.groupby(["Date"])["isAccepted"]
    .mean()
    .reset_index()
)

fig = px.scatter(
    observed_daily_accepts,
    x="Date",
    y="isAccepted"
)
fig.update_traces(marker={"size": 5})
fig.update_layout(yaxis_tickformat=".0%")
fig.show()